# Pendulum Energy — KE vs PE

A pendulum bob of mass $m$ on a string of length $L$ is released from rest at an angle $\theta_0$.

- **Left:** the pendulum swinging (the dotted line marks the release height — the bob always rises back to it).
- **Right:** the energies plotted **against angular position**. The dots slide back and forth along their curves in time with the bob.

The three curves:

- **Blue — potential energy**, $PE = mgh = mgL(1 - \cos\theta)$: greatest at the extremes, zero at the bottom.
- **Green — kinetic energy**, $KE = \tfrac{1}{2}mv^2$: zero at the extremes, greatest at the bottom.
- **Red — total energy**, $E = PE + KE$: a flat line — energy is **conserved** (no friction).

At any position, the blue and green dots always add up to the red line. At the release point it's all PE; at the bottom it's all KE; in between, the energy is continuously swapping between the two.

**Things to try:**

- Bigger release angle → the curves get taller ($E = mgL(1-\cos\theta_0)$) but the total still stays flat.
- Heavier mass → more energy, but the **motion is unchanged** — the period doesn't depend on mass.
- Longer string → slower swing: $T \approx 2\pi\sqrt{L/g}$ (the label also shows the exact period — the approximation is only perfect for small swings).


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Circle

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML





g = 9.81   # m/s^2





# ------------------------------------------------------------

# The physics

# ------------------------------------------------------------



def quarter_swing(L, th_max_deg, n=2000):

    """Exact (t, theta) pairs for the quarter swing release -> bottom.

    Uses the standard substitution sin(theta/2) = sin(th0/2) * sin(A), which

    turns dt = dth/omega into the smooth elliptic integral

        t(theta) = sqrt(L/g) * integral from A(theta) to pi/2 of

                   dA / sqrt(1 - k^2 sin^2 A),   k = sin(th0/2)

    evaluated by the trapezoid rule (the integrand has no singularities).

    """

    th0 = np.radians(th_max_deg)

    k = np.sin(th0 / 2)

    ths = np.linspace(th0, 0.0, 401)                    # theta grid, release -> bottom

    phis = np.arcsin(np.clip(np.sin(ths / 2) / k, -1.0, 1.0))

    alphas = np.linspace(0, np.pi / 2, n)

    integrand = 1.0 / np.sqrt(1 - k**2 * np.sin(alphas)**2)

    dA = alphas[1] - alphas[0]

    F_all = np.concatenate([[0.0],

                            np.cumsum(0.5 * (integrand[:-1] + integrand[1:]) * dA)])

    F_of_phi = np.interp(phis, alphas, F_all)   # F(phi, k) for each theta

    K = F_all[-1]                               # complete integral K(k)

    ts = np.sqrt(L / g) * (K - F_of_phi)        # t = 0 at release

    return ts, ths





def pendulum_period(L, th_max_deg):

    """Exact period of an ideal pendulum = 4 x quarter swing."""

    ts, _ = quarter_swing(L, th_max_deg)

    return 4.0 * ts[-1]





def energies(m, L, th_deg, th_max_deg):

    """PE, KE and total E at angle th, released from rest at th_max."""

    E = m * g * L * (1 - np.cos(np.radians(th_max_deg)))

    PE = m * g * L * (1 - np.cos(np.radians(th_deg)))

    KE = E - PE

    return PE, KE, E





def swing_angles(L, th_max_deg, n_frames):

    """theta (degrees) at n_frames uniform times over one full period.

    One quarter release -> bottom is computed exactly; the other three

    quarters are time-reversed / mirrored copies of it.

    """

    ts_q, ths_q = quarter_swing(L, th_max_deg)   # release -> bottom, t: 0 -> t_q

    t_q = ts_q[-1]

    ts = np.linspace(0, 4 * t_q, n_frames, endpoint=False)

    tau = ts % t_q                               # time into the current quarter

    which = np.floor(ts / t_q).astype(int)       # which quarter (0..3)



    ths = np.empty_like(ts)

    m = which == 0                               # release -> bottom

    ths[m] = np.interp(tau[m], ts_q, ths_q)

    m = which == 1                               # bottom -> far side (reversed)

    ths[m] = -np.interp(t_q - tau[m], ts_q, ths_q)

    m = which == 2                               # far side -> bottom

    ths[m] = -np.interp(tau[m], ts_q, ths_q)

    m = which == 3                               # bottom -> release (reversed)

    ths[m] = np.interp(t_q - tau[m], ts_q, ths_q)



    return np.degrees(ths)





# ------------------------------------------------------------

# Drawing

# ------------------------------------------------------------



def build_pendulum_figure(m, L, th_max_deg):

    """Two panels: the pendulum (left) and the energy plot (right)."""

    th0 = np.radians(th_max_deg)

    E = m * g * L * (1 - np.cos(th0))

    bob_r = 0.07 * L * m ** (1 / 3)



    fig, (axp, axe) = plt.subplots(

        1, 2, figsize=(11, 4.6), gridspec_kw=dict(width_ratios=[1, 1.35]))



    # ---------------- left: the pendulum ----------------

    R = 1.22 * L

    axp.set_xlim(-R, R)

    axp.set_ylim(-R, 0.32 * L)

    axp.set_aspect("equal")



    # support and pivot

    axp.plot([-0.22 * L, 0.22 * L], [0, 0], color="black", linewidth=3)

    for xh in np.linspace(-0.22 * L, 0.22 * L, 7):

        axp.plot([xh, xh - 0.06 * L], [0, 0.09 * L], color="black", linewidth=1)

    axp.plot([0], [0], "o", color="black", markersize=5)



    # vertical reference, swing path arc, release height, ghost bob

    axp.plot([0, 0], [0, -1.08 * L], ls=":", color="gray", linewidth=1)

    arc_ths = np.linspace(-th0, th0, 100)

    axp.plot(L * np.sin(arc_ths), -L * np.cos(arc_ths),

             color="lightgray", linewidth=1, zorder=1)

    axp.axhline(-L * np.cos(th0), color="gray", ls=":", linewidth=1.2, zorder=1)

    axp.add_patch(Circle((L * np.sin(th0), -L * np.cos(th0)), bob_r,

                         facecolor="none", edgecolor="gray", ls=":", zorder=1))



    string = axp.plot([], [], color="black", linewidth=1.5, zorder=2)[0]

    bob = Circle((L * np.sin(th0), -L * np.cos(th0)), bob_r,

                 facecolor="lightblue", edgecolor="black", linewidth=2, zorder=3)

    axp.add_patch(bob)



    info_p = axp.text(0.03, 0.96, "", transform=axp.transAxes,

                      ha="left", va="top", fontsize=11,

                      bbox=dict(facecolor="white", alpha=0.85,

                                edgecolor="lightgray"))



    axp.set_title(f"m = {m:g} kg,  L = {L:g} m,  released from θ₀ = {th_max_deg:g}°",

                  fontsize=12)

    axp.set_xticks([])

    axp.set_yticks([])

    for side in ("left", "right", "top", "bottom"):

        axp.spines[side].set_visible(False)



    # ---------------- right: energy vs position ----------------

    ths = np.linspace(-th_max_deg, th_max_deg, 200)

    PE_curve = m * g * L * (1 - np.cos(np.radians(ths)))

    KE_curve = E - PE_curve



    axe.plot(ths, PE_curve, color="tab:blue", linewidth=2.2)

    axe.plot(ths, KE_curve, color="tab:green", linewidth=2.2)

    axe.plot([-th_max_deg, th_max_deg], [E, E], color="tab:red", linewidth=2.2)



    # curve labels in clear spots (no legend box to collide with the dots)

    axe.text(0, 0.06 * E, "PE", color="tab:blue", fontsize=12,

             fontweight="bold", ha="center", va="center")

    axe.text(0, 0.88 * E, "KE", color="tab:green", fontsize=12,

             fontweight="bold", ha="center", va="center")

    axe.text(th_max_deg - 2, 1.05 * E, "E = PE + KE", color="tab:red",

             fontsize=12, fontweight="bold", ha="right", va="center")



    pe_dot = axe.plot([], [], "o", markersize=11, color="tab:blue",

                      markeredgecolor="black", zorder=5)[0]

    ke_dot = axe.plot([], [], "o", markersize=11, color="tab:green",

                      markeredgecolor="black", zorder=5)[0]

    e_dot = axe.plot([], [], "o", markersize=11, color="tab:red",

                     markeredgecolor="black", zorder=5)[0]

    vline = axe.axvline(th_max_deg, color="gray", linewidth=1, alpha=0.6, zorder=2)



    info_e = axe.text(0.5, 0.97, "", transform=axe.transAxes,

                      ha="center", va="top", fontsize=11,

                      bbox=dict(facecolor="white", alpha=0.85,

                                edgecolor="lightgray"))



    axe.set_xlim(-th_max_deg - 6, th_max_deg + 6)

    axe.set_ylim(0, 1.18 * E)

    axe.set_xlabel("Angular position θ (°)")

    axe.set_ylabel("Energy (J)")

    axe.set_title(f"Energy vs position — total is always {E:.3g} J", fontsize=12)

    for side in ("right", "top"):

        axe.spines[side].set_visible(False)



    fig.tight_layout()



    artists = dict(

        string=string, bob=bob, info_p=info_p,

        pe_dot=pe_dot, ke_dot=ke_dot, e_dot=e_dot,

        vline=vline, info_e=info_e,

    )

    return fig, artists, bob_r





def set_state(art, m, L, th_max_deg, th_deg, bob_r):

    """Move every artist to match angle th_deg."""

    th = np.radians(th_deg)

    th0 = np.radians(th_max_deg)



    x = L * np.sin(th)

    y = -L * np.cos(th)

    art["string"].set_data([0, x], [0, y])

    art["bob"].center = (x, y)



    PE, KE, E = energies(m, L, th_deg, th_max_deg)

    art["pe_dot"].set_data([th_deg], [PE])

    art["ke_dot"].set_data([th_deg], [KE])

    art["e_dot"].set_data([th_deg], [E])

    art["vline"].set_xdata([th_deg])



    h = L * (1 - np.cos(th))                      # height above the bottom

    v = np.sqrt(max(2 * g * L * (np.cos(th) - np.cos(th0)), 0.0))

    art["info_p"].set_text(f"θ = {th_deg:+.1f}°\nh = {h:.3g} m\nv = {v:.3g} m/s")

    art["info_e"].set_text(

        f"PE = {PE:.3g} J      KE = {KE:.3g} J      PE + KE = {PE + KE:.3g} J")





def show_release(m, L, th_max_deg):

    """Static preview: pendulum at the release point, dots at the start."""

    fig, art, bob_r = build_pendulum_figure(m, L, th_max_deg)

    set_state(art, m, L, th_max_deg, th_max_deg, bob_r)

    display(fig)

    plt.close(fig)





def pendulum_animation(m, L, th_max_deg, n_frames=64):

    """In-browser animation: one full swing, looping in real time."""

    fig, art, bob_r = build_pendulum_figure(m, L, th_max_deg)

    ths = swing_angles(L, th_max_deg, n_frames)

    T = pendulum_period(L, th_max_deg)

    interval = min(max(1000.0 * T / n_frames, 30), 80)   # ~ real time



    def update(i):

        set_state(art, m, L, th_max_deg, ths[i], bob_r)

        return tuple(art.values())



    anim = FuncAnimation(fig, update, frames=n_frames,

                         interval=interval, repeat=True)

    plt.close(fig)

    return HTML(anim.to_jshtml())





def summary(m, L, th_max_deg):

    E = m * g * L * (1 - np.cos(np.radians(th_max_deg)))

    v_max = np.sqrt(2 * g * L * (1 - np.cos(np.radians(th_max_deg))))

    T0 = 2 * np.pi * np.sqrt(L / g)

    T = pendulum_period(L, th_max_deg)

    return (f"<b>E = mgL(1 − cosθ₀)</b> = ({m:g})(9.81)({L:g})(1 − cos {th_max_deg:g}°) "

            f"= <b>{E:.3g} J</b><br>"

            f"Release: all PE &nbsp;·&nbsp; Bottom: all KE, v_max = <b>{v_max:.3g} m/s</b><br>"

            f"Period: exact T = <b>{T:.2f} s</b> "

            f"(small-angle approx 2π√(L/g) = {T0:.2f} s)")


In [ ]:
# ============================================================

# Pendulum energy: set mass, length, release angle, then Run

# ============================================================



mass_slider = widgets.FloatSlider(

    value=1.0, min=0.2, max=2.0, step=0.2,

    description="m (kg):", continuous_update=False

)

length_slider = widgets.FloatSlider(

    value=1.0, min=0.5, max=2.0, step=0.1,

    description="L (m):", continuous_update=False

)

angle_slider = widgets.FloatSlider(

    value=60, min=5, max=90, step=5,

    description="θ₀ (°):", continuous_update=False

)



run_button = widgets.Button(description="Run", button_style="primary", icon="play")

reset_button = widgets.Button(description="Reset")

value_label = widgets.HTML()

sim_output = widgets.Output(layout=widgets.Layout(width="1000px"))





def show_preview():

    value_label.value = summary(mass_slider.value,

                                length_slider.value,

                                angle_slider.value)

    show_release(mass_slider.value, length_slider.value, angle_slider.value)





def run_clicked(button):

    value_label.value = summary(mass_slider.value,

                                length_slider.value,

                                angle_slider.value)

    with sim_output:

        sim_output.clear_output(wait=True)

        display(pendulum_animation(mass_slider.value,

                                   length_slider.value,

                                   angle_slider.value))





def reset_clicked(button):

    with sim_output:

        sim_output.clear_output(wait=True)

        show_preview()





def slider_changed(change):

    with sim_output:

        sim_output.clear_output(wait=True)

        show_preview()





run_button.on_click(run_clicked)

reset_button.on_click(reset_clicked)

mass_slider.observe(slider_changed, names="value")

length_slider.observe(slider_changed, names="value")

angle_slider.observe(slider_changed, names="value")



controls = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    mass_slider,

    length_slider,

    angle_slider,

    widgets.HBox([run_button, reset_button]),

    value_label

], layout=widgets.Layout(width="330px"))



display(widgets.HBox([controls, sim_output],

                     layout=widgets.Layout(align_items="flex-start", gap="18px")))



with sim_output:

    show_preview()
